# Model Evaluation and Comparison

This notebook analyzes and compares the performance of four trained models:

- **DenseNet – Fine-tuned**
- **DenseNet – Frozen Backbone**
- **ResNet – Fine-tuned**
- **ResNet – Frozen Backbone**

Before running this notebook, you must first **train and evaluate all four models** using the following notebooks:

- [densenet_finetuned.ipynb](./densenet_finetuned.ipynb)
- [densenet_frozen.ipynb](./densenet_frozen.ipynb)
- [resnet_finetuned.ipynb](./resnet_finetuned.ipynb)
- [resnet_frozen.ipynb](./resnet_frozen.ipynb)

Running these training notebooks will generate all required metrics such as:
- training logs
- test predictions
- final results (loss, accuracy, AUCs)

These metrics are automatically saved under the `metrics/` directory.  
This analysis notebook then loads those saved files and generates plots, comparisons, and insights.


In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import roc_curve, auc

In [2]:
BASE_DIR = "../metrics"
PLOT_DIR = "../plots"
NUM_CLASSES = 14

MODEL_CONFIGS = {
    "densenet_finetuning": {
        "path": os.path.join("densenet", "fine_tuning")
    },
    "densenet_frozen": {
        "path": os.path.join("densenet", "frozen_backbone")
    },
    "resenet_finetuning": {
        "path": os.path.join("resnet", "fine_tuning")
    },
    "resenet_frozen": {
        "path": os.path.join("resnet", "frozen_backbone")
    }
}

In [3]:
def load_metrics(model_key):
    model_path = MODEL_CONFIGS[model_key]["path"]

    training_log = pd.read_csv(os.path.join(BASE_DIR, model_path, "training_log.csv"))
    final_results = pd.read_csv(os.path.join(BASE_DIR, model_path, "final_results.csv"))
    test_preds = pd.read_csv(os.path.join(BASE_DIR, model_path, "test_predictions.csv"))

    return training_log, final_results, test_preds

## Training Curves

For each model, we plot three training-time metrics across epochs:

1. **Training Loss** – how well the model fits the training data  
2. **Validation Loss** – how well the model generalizes to unseen data  
3. **Validation Macro AUC** – class-balanced validation performance

These plots help visualize convergence, overfitting, and the effect of fine-tuning vs. freezing the backbone.


In [4]:
def plot_training_curves(model_key, training_log):
    fig, ax = plt.subplots(3, 1, figsize=(7, 15))

    ax[0].plot(training_log["epoch"], training_log["train_loss"])
    ax[0].set_title(f"{model_key}: Training Loss")
    ax[0].set_xlabel("Epoch")
    ax[0].set_ylabel("Training Loss")

    ax[1].plot(training_log["epoch"], training_log["val_loss"])
    ax[1].set_title(f"{model_key}: Validation Loss")
    ax[1].set_xlabel("Epoch")
    ax[1].set_ylabel("Val Loss")

    ax[2].plot(training_log["epoch"], training_log["val_macro_auc"])
    ax[2].set_title(f"{model_key}: Validation Macro AUC")
    ax[2].set_xlabel("Epoch")
    ax[2].set_ylabel("AUC")

    save_path = os.path.join(PLOT_DIR, MODEL_CONFIGS[model_key]["path"], "training_curves.png")
    plt.tight_layout()
    plt.savefig(save_path)
    plt.close()

## Macro ROC Curve

To evaluate each model's ability to distinguish between positive and negative cases across all 14 disease classes, we compute and plot the **Macro ROC Curve**.

### How it works:
- For every class, we compute:
  - False Positive Rate (FPR)
  - True Positive Rate (TPR)
  - ROC AUC
- We then aggregate these by:
  - Taking the **union of all FPR points**
  - Interpolating TPR values for each class
  - Averaging them to obtain a **macro-averaged TPR**
- Finally, we compute the **macro AUC** and plot the ROC curve.

This provides a **class-balanced comparison** between:
- DenseNet vs. ResNet  
- Fine-tuning vs. Frozen backbone


In [5]:
def plot_macro_roc(model_key, test_preds):
    fpr_dict = {}
    tpr_dict = {}
    auc_dict = {}

    for i in range(NUM_CLASSES):
        y_true = test_preds[f"true_{i}"].values
        y_score = test_preds[f"prob_{i}"].values

        fpr, tpr, _ = roc_curve(y_true, y_score)
        roc_auc = auc(fpr, tpr)

        fpr_dict[i] = fpr
        tpr_dict[i] = tpr
        auc_dict[i] = roc_auc

    all_fprs = np.unique(np.concatenate([fpr_dict[i] for i in range(NUM_CLASSES)]))
    mean_tpr = np.zeros_like(all_fprs)

    for i in range(NUM_CLASSES):
        mean_tpr += np.interp(all_fprs, fpr_dict[i], tpr_dict[i])

    mean_tpr /= NUM_CLASSES
    macro_auc = auc(all_fprs, mean_tpr)

    plt.figure(figsize=(7, 6))
    plt.plot(all_fprs, mean_tpr, lw=2, label=f"Macro ROC (AUC = {macro_auc:.4f})")
    plt.plot([0, 1], [0, 1], linestyle="--")

    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(f"Macro ROC Curve - {model_key}")
    plt.legend(loc="lower right")

    save_path = os.path.join(PLOT_DIR, MODEL_CONFIGS[model_key]["path"], "macro_roc.png")
    plt.savefig(save_path)
    plt.close()

## Per-Class AUC Bar Plot

To understand how each model performs on individual disease classes, we generate a **Per-Class AUC Bar Plot**.

### What this plot shows:
- Each class (C0–C13) corresponds to one pathology from the NIH Chest X-ray dataset.
- For every class, we plot the AUC achieved by:
  - DenseNet (Fine-tuned)
  - DenseNet (Frozen)
  - ResNet (Fine-tuned)
  - ResNet (Frozen)
- Bars are grouped by class to make comparisons easier.

### Why this is useful:
- Highlights strengths and weaknesses of each model on specific diseases.
- Shows whether fine-tuning consistently improves AUC over frozen backbone.
- Reveals classes that are generally harder to classify regardless of architecture.


In [6]:
def plot_per_class_auc_bar(model_aucs):
    plt.figure(figsize=(16, 6))

    class_indices = range(NUM_CLASSES)
    bar_width = 0.18

    for idx, (model_key, aucs) in enumerate(model_aucs.items()):
        plt.bar([c + idx * bar_width for c in class_indices], aucs, width=bar_width, label=model_key)

    plt.xticks([c + 1.5 * bar_width for c in class_indices], [f"C{i}" for i in class_indices])
    plt.xlabel("Class Index")
    plt.ylabel("AUC")
    plt.title("Per-Class AUC Comparison Across Models")
    plt.legend()
    plt.grid(axis="y", alpha=0.3)

    save_path = os.path.join(PLOT_DIR, "per_class_auc_bar.png")
    plt.tight_layout()
    plt.savefig(save_path)
    plt.close()

## Per-Class AUC Heatmap

To visualize class-wise performance differences more compactly, we generate a **Per-Class AUC Heatmap** using the AUC values from all models.

### What this heatmap shows:
- Rows correspond to the four models:
  - DenseNet (Fine-tuned)
  - DenseNet (Frozen)
  - ResNet (Fine-tuned)
  - ResNet (Frozen)
- Columns correspond to the 14 disease classes (C0–C13).
- Each cell contains the **AUC score** for that model on that class.
- Color intensity indicates performance (darker = higher AUC).

### Why this is useful:
- Provides an overview of which models perform well or poorly across different pathologies.
- Makes it easy to identify:
  - consistent strong/weak classes
  - models that are uniformly better
  - differences between fine-tuning and freezing
- More compact and interpretable than bar charts for 14 classes × 4 models.


In [7]:
def plot_per_class_auc_heatmap(auc_df):
    plt.figure(figsize=(16, 5))
    sns.heatmap(auc_df, annot=True, fmt=".4f", cmap="viridis")
    plt.title("Per-Class AUC Heatmap Across Models")
    plt.xlabel("Class")
    plt.ylabel("Model")

    save_path = os.path.join(PLOT_DIR, "per_class_auc_heatmap.png")
    plt.tight_layout()
    plt.savefig(save_path)
    plt.close()

In [9]:
model_aucs = {}

for model_key in MODEL_CONFIGS:
    training_log, final_results, test_preds = load_metrics(model_key)
    model_aucs[model_key] = [
        final_results[f"class_{i}_auc"].values[0] for i in range(NUM_CLASSES)
    ]
    plot_training_curves(model_key, training_log)
    plot_macro_roc(model_key, test_preds)

auc_df = pd.DataFrame(
    [model_aucs[model_key] for model_key in MODEL_CONFIGS],
    list(MODEL_CONFIGS.keys()),
    columns=[f"C{i}" for i in range(NUM_CLASSES)],
)

plot_per_class_auc_bar(model_aucs)
plot_per_class_auc_heatmap(auc_df)